# Solving CartPole-v1 with PPO (from scratch)

This notebook demonstrates how to solve the CartPole-v1 environment using a custom implementation of the Proximal Policy Optimization (PPO) algorithm in PyTorch. No external RL libraries are used.

---

**Note:** All environment and hyperparameter settings are now collected in a single `CONFIG` dictionary at the top of the notebook. To change the environment or any hyperparameter, simply edit the values in the config cell.

## 1. Install and Import Required Libraries
We will use gymnasium and torch for this implementation.

In [1]:
%pip install gymnasium stable-baselines3 wandb tsilva-notebook-utils==0.0.114 --quiet

Note: you may need to restart the kernel to use updated packages.


In [2]:
from tsilva_notebook_utils.colab import load_secrets_into_env

_ = load_secrets_into_env([
    'WANDB_API_KEY'
])

In [3]:
from tsilva_notebook_utils.torch import get_default_device
DEVICE = get_default_device()
DEVICE

device(type='mps')

## 2. Set Up CartPole-v1 Environment
We will initialize the CartPole-v1 environment and display its basic information.

In [ ]:
import torch.nn as nn
from tsilva_notebook_utils.gymnasium import build_env as _build_env, set_random_seed

# --- Config dictionary for all hyperparameters and environment settings ---
def setup_config(env_id):
    common = dict(
        env_id=env_id,         # Environment name
        max_epochs=-1,          # Maximum number of training epochs (-1 for no limit)
        seed=42,               # Random seed for reproducibility
        gamma=0.99,            # Discount factor for future rewards
        lam=0.95,              # GAE lambda for advantage estimation
        clip_epsilon=0.2,      # PPO clip range for policy update
        minibatch_size=64,     # Minibatch size for SGD
        train_episodes=16,     # Number of episodes to collect per training step
        eval_interval=5,       # Evaluate every N epochs
        eval_episodes=16,      # Number of episodes for evaluation
        reward_threshold=200,  # Reward threshold to consider environment solved
        policy_lr=3e-4,        # Learning rate for policy network
        value_lr=1e-3,         # Learning rate for value network
        hidden_dim=64,         # Hidden layer size for networks
        entropy_coef=0.01,     # Coefficient for entropy bonus (encourages exploration)
        normalize=False,       # Whether to use input normalization
        mean_reward_window=100,  # Window size for mean reward calculation (for early stopping)
        n_envs="auto"          # Maximum number of parallel environments
    )
    env_specific = {
        "CartPole-v1": dict(
            gamma=0.99,           # Standard discount - CartPole episodes are short
            lam=0.95,             # GAE lambda - good bias/variance tradeoff
            clip_epsilon=0.2,     # Standard PPO clip - CartPole is stable
            minibatch_size=64,    # Smaller than current 256 for more frequent updates
            eval_interval=5,      # Less frequent eval - CartPole converges quickly
            eval_episodes=16,     # Sufficient for stable eval estimates
            reward_threshold=475, # Official CartPole-v1 solved threshold
            policy_lr=3e-4,       # Standard PPO learning rate
            value_lr=3e-4,        # Match policy LR for balanced learning
            hidden_dim=64,        # Sufficient capacity for CartPole's simple state space
            entropy_coef=0.01,    # Low entropy - CartPole doesn't need much exploration
            train_episodes=32,    # More episodes per update for better sample efficiency
            #normalize=True,       # Helps with CartPole's varying state magnitudes
            mean_reward_window=100 # Standard window for CartPole evaluation
        ),
        "LunarLander-v3": dict(
            gamma=0.99,           # Standard discount for LunarLander
            lam=0.95,             # GAE lambda, balances bias/variance
            clip_epsilon=0.2,     # PPO clip range, stable for LunarLander
            minibatch_size=64,    # Larger batch for more stable updates
            eval_interval=2,      # Evaluate every 2 epochs
            reward_threshold=200, # Solved threshold for LunarLander-v3
            policy_lr=1e-4,       # Lower LR for more complex env
            value_lr=5e-4,        # Lower LR for value net
            hidden_dim=128,       # Larger net for more complex env
            entropy_coef=0.02     # Higher entropy for more exploration
        ),
        "Acrobot-v1": dict(
            gamma=0.99,           # Standard discount for Acrobot
            lam=0.95,             # GAE lambda, balances bias/variance
            clip_epsilon=0.2,     # PPO clip range, stable for Acrobot
            minibatch_size=32,    # Smaller batch for faster updates
            eval_interval=2,      # Evaluate more frequently for fast convergence
            reward_threshold=-100, # Solved threshold for Acrobot-v1 (average reward > -100)
            policy_lr=3e-4,       # Standard PPO learning rate
            value_lr=1e-3,        # Slightly higher for value net
            hidden_dim=64,        # Small net is sufficient for Acrobot
            entropy_coef=0.01     # Typical entropy for Acrobot
        ),
        "Pendulum-v1": dict(
            gamma=0.99,           # Standard discount for Pendulum
            lam=0.95,             # GAE lambda, balances bias/variance
            clip_epsilon=0.2,     # PPO clip range, stable for Pendulum
            minibatch_size=64,    # Larger batch for continuous action
            eval_interval=2,      # Evaluate every 2 epochs
            eval_episodes=5,      # Fewer eval episodes for speed
            reward_threshold=-200, # Solved threshold for Pendulum-v1 (average reward > -200)
            policy_lr=3e-4,       # Standard PPO learning rate
            value_lr=1e-3,        # Slightly higher for value net
            hidden_dim=128,       # Larger net for continuous control
            entropy_coef=0.0      # No entropy for deterministic continuous control
        ),
        "MountainCar-v0": dict(
            gamma=0.99,             # Discount factor (keep)
            lam=0.97,               # Slightly higher GAE lambda for more bias reduction
            clip_epsilon=0.15,      # Tighter PPO clip for more stable updates
            minibatch_size=16,      # Smaller minibatch for more frequent updates
            eval_interval=2,        # Keep frequent evaluation
            eval_episodes=10,       # Keep
            reward_threshold=-110,  # Keep
            policy_lr=1e-4,         # Lower learning rate for more stable policy updates
            value_lr=5e-4,          # Lower value net LR for stability
            hidden_dim=128,         # Larger network for more capacity
            entropy_coef=0.05       # Higher entropy for hard exploration
        ),
    }

    if env_id not in env_specific:
        raise ValueError(f"Unsupported env_id: {env_id}")
    config = {**common, **env_specific[env_id]}

    # TODO: encapsulate this
    import multiprocessing
    n_envs = config['n_envs']
    train_episodes = config['train_episodes']
    eval_episodes = config['eval_episodes']
    if n_envs == "auto": n_envs = multiprocessing.cpu_count()
    if train_episodes % n_envs != 0:
        raise ValueError(f"train_episodes ({config['train_episodes']}) must be divisible by n_envs ({n_envs})")
    if eval_episodes % n_envs != 0:
        raise ValueError(f"eval_episodes ({config['eval_episodes']}) must be divisible by n_envs ({n_envs})")
    
    return config

ENV_ID = "CartPole-v1"
#ENV_ID = "Acrobot-v1"
#ENV_ID = "LunarLander-v3"
#ENV_ID = "Pendulum-v1"
#ENV_ID = "MountainCar-v0"
CONFIG = setup_config(ENV_ID)
CONFIG

{'env_id': 'CartPole-v1',
 'max_epochs': -1,
 'seed': 42,
 'gamma': 0.99,
 'lam': 0.95,
 'clip_epsilon': 0.2,
 'minibatch_size': 64,
 'train_episodes': 32,
 'eval_interval': 10,
 'eval_episodes': 16,
 'reward_threshold': 475,
 'policy_lr': 0.0003,
 'value_lr': 0.0003,
 'hidden_dim': 64,
 'entropy_coef': 0.01,
 'normalize': False,
 'mean_reward_window': 100,
 'n_envs': 'auto'}

In [5]:
from tsilva_notebook_utils.gymnasium import log_env_info

# Set random seed for reproducibility
set_random_seed(CONFIG['seed'])

# Wrap build env with config parameters
build_env = lambda seed, n_envs=None: _build_env(
    CONFIG['env_id'], 
    norm_obs=CONFIG['normalize'], 
    n_envs=n_envs if n_envs is not None else CONFIG['n_envs'], 
    seed=seed
)

# Test building env
env = build_env(CONFIG['seed'])
log_env_info(env)

Environment Info (SubprocVecEnv with 8 envs)
  Env ID: CartPole-v1
  Observation space: Box(low=[-4.8, -inf, -0.419, -inf], high=[4.8, inf, 0.419, inf], shape=(4,), dtype=float32)
  Action space: Discrete(2)
  Max episode steps: 500


## 3. Implement PPO Agent
We will define the policy and value networks, and the PPO update step.

In [6]:
class PolicyNet(nn.Module):
    def __init__(self, obs_dim, act_dim, hidden_dim=None):
        super().__init__()
        h = hidden_dim or CONFIG['hidden_dim']
        self.net = nn.Sequential(
            nn.Linear(obs_dim, h), nn.Tanh(),
            nn.Linear(h, h), nn.Tanh(),
            nn.Linear(h, act_dim)
        )
    def forward(self, x):
        return self.net(x)

class ValueNet(nn.Module):
    def __init__(self, obs_dim, hidden_dim=None):
        super().__init__()
        h = hidden_dim or CONFIG['hidden_dim']
        self.net = nn.Sequential(
            nn.Linear(obs_dim, h), nn.Tanh(),
            nn.Linear(h, h), nn.Tanh(),
            nn.Linear(h, 1)
        )
    def forward(self, x):
        return self.net(x)

In [ ]:
import time
import torch
import multiprocessing
import numpy as np
import pytorch_lightning as pl
from torch.utils.data import DataLoader
from torch.distributions import Categorical
from collections import deque
from tsilva_notebook_utils.gymnasium import RolloutDataset, collect_rollouts, group_trajectories_by_episode

# ---------------------------------------------------------------------
#  PPO Lightning module
#  (assumes PolicyNet, ValueNet, build_env, collect_rollouts are defined)
# ---------------------------------------------------------------------
class PPOAgent(pl.LightningModule):
    def __init__(self, obs_dim, act_dim, config):
        super().__init__()
        self.save_hyperparameters()

        # ----------------- unpack config -----------------
        self.config = config
        self.entropy_coef   = config['entropy_coef']
        self.clip_epsilon   = config['clip_epsilon']
        self.gamma          = config['gamma']
        self.lam            = config['lam']
        self.minibatch_size     = config['minibatch_size']
        self.train_episodes    = config['train_episodes'] 
        self.eval_interval      = config['eval_interval']
        self.eval_episodes      = config['eval_episodes']
        self.reward_threshold   = config['reward_threshold']
        self.policy_lr          = config['policy_lr']
        self.value_lr           = config['value_lr']
        self.mean_reward_window = config['mean_reward_window']

        # ----------------- models & env -----------------
        self.policy_model = PolicyNet(
            obs_dim, act_dim,
            hidden_dim=config['hidden_dim']# TODO: read from params
        )
        self.value_model = ValueNet(
            obs_dim,
            hidden_dim=config['hidden_dim']# TODO: read from params
        )
        self.env = build_env(config['seed']) # TODO: read from params
        self.obs_dim, self.act_dim = obs_dim, act_dim

        # ----------------- rollout storage --------------
        self.rollout_ds = RolloutDataset()   # <-- created once

        # TODO: softcode
        self.episode_reward_deque = deque(maxlen=self.mean_reward_window)  # Store recent mean rewards for early stopping
        
        # Disable automatic optimization to allow maintaining a 
        # different optimizer for each model (policy and value)
        self.automatic_optimization = False


        self.training_start_time = None
        self.training_end_time = None

    # ===================================================
    #  Lightning hooks
    # ===================================================
    def setup(self, stage: str):
        """Collect an initial roll-out before dataloaders are requested."""
        if stage == "fit": self._collect_and_store_rollout()

    # TODO: is this called only once?
    def train_dataloader(self):
        """Standard DataLoader built *once*; dataset is mutable."""
        return DataLoader(
            self.rollout_ds,
            batch_size=self.minibatch_size,
            shuffle=True,
            pin_memory=True if self.device.type != 'mps' else False,  # Set to False for MPS compatibility
            # TODO: doesn't converge when True (memory sharing issues?)
            #persistent_workers=False,
            num_workers=multiprocessing.cpu_count() // 2 if self.device.type != 'mps' else 0
        )

    
    def on_fit_start(self):
        """Called when training starts"""
        self.training_start_time = time.time()
        print(f"PPO training started at {time.strftime('%Y-%m-%d %H:%M:%S')}")
    
    def on_fit_end(self):
        """Called when training ends"""
        self.training_end_time = time.time()
        total_time = self.training_end_time - self.training_start_time
        print(f"PPO training completed in {total_time:.2f} seconds ({total_time/60:.2f} minutes)")

    def on_train_epoch_start(self):
        """Refresh roll-out tensors in-place each epoch."""
        self._collect_and_store_rollout()

    def on_train_epoch_end(self):
        """Evaluate model every N epochs and check for early stopping."""
        if (self.current_epoch + 1) % self.eval_interval == 0:
            eval_mean_reward = self._evaluate_model()
            
            self.log('eval/mean_reward', eval_mean_reward, prog_bar=True)
            
            # Check for early stopping based on eval reward
            if eval_mean_reward >= self.reward_threshold:
                print(f"Early stopping at epoch {self.current_epoch} with eval mean reward {eval_mean_reward:.2f} >= threshold {self.reward_threshold}")
                self.trainer.should_stop = True

    # ---------------------------------------------------
    def training_step(self, batch, batch_idx):
        opt_policy, opt_value = self.optimizers()

        # unpack batch
        (states, actions, rewards, dones, old_logps, values, advantages, returns, frames) = batch

        # main PPO loop
        policy_losses, value_losses = [], []
        clip_fractions, entropy_vals, kl_divs, approx_kl_divs = [], [], [], []
    
        logits = self.policy_model(states)
        dist   = Categorical(logits=logits)
        new_logps = dist.log_prob(actions)

        ratio = torch.exp(new_logps - old_logps)
        surr1 = ratio * advantages
        surr2 = torch.clamp(
            ratio, 1.0 - self.clip_epsilon, 1.0 + self.clip_epsilon
        ) * advantages
        entropy = dist.entropy().mean()

        # === PPO DEBUG METRICS ===
        # 1. Clipping fraction - how often is clipping happening?
        clip_fraction = ((ratio < 1.0 - self.clip_epsilon) | (ratio > 1.0 + self.clip_epsilon)).float().mean()
        
        # 2. KL divergence between old and new policies
        # Use to detect when policy updates are too large
        # Thresholds depend on environment, batch size and reward scale
        # As a rule of thumb, for discrete actions, a KL of 0.01-0.1 is reasonable,
        # while for continuous actions, it can be higher (0.1-0.5).
        kl_div = (old_logps - new_logps).mean()  # Simple KL approximation
        approx_kl = ((ratio - 1) - torch.log(ratio)).mean()  # Better KL approximation
        
        # 3. Explained variance of value function
        value_pred = self.value_model(states).squeeze()

        # Explained variance measures how well value predictions match returns 
        # (1 = perfect, 0 = no correlation, <0 = worse than random).
        explained_var = 1 - torch.var(returns - value_pred) / torch.var(returns)

        policy_loss = -torch.min(surr1, surr2).mean()
        policy_loss -= self.entropy_coef * entropy
        value_loss = 0.5 * ((returns - value_pred) ** 2).mean()

        # Store metrics
        clip_fractions.append(clip_fraction.detach())
        entropy_vals.append(entropy.detach())
        kl_divs.append(kl_div.detach())
        approx_kl_divs.append(approx_kl.detach())

        # ---------- optimizers ----------
        opt_policy.zero_grad()
        self.manual_backward(policy_loss)
        opt_policy.step()

        opt_value.zero_grad()
        self.manual_backward(value_loss)
        opt_value.step()

        policy_losses.append(policy_loss.detach())
        value_losses.append(value_loss.detach())

        # Calculate means
        mean_policy_loss = torch.stack(policy_losses).mean()
        mean_value_loss = torch.stack(value_losses).mean()
        mean_clip_fraction = torch.stack(clip_fractions).mean()
        mean_entropy = torch.stack(entropy_vals).mean()
        mean_kl = torch.stack(kl_divs).mean()
        mean_approx_kl = torch.stack(approx_kl_divs).mean()

        # Additional useful metrics
        advantage_mean = advantages.mean()
        advantage_std = advantages.std()
        value_mean = values.mean()
        returns_mean = returns.mean()

        mean_reward = np.mean(self.episode_reward_deque) if len(self.episode_reward_deque) >= self.mean_reward_window else None

        # Enhanced logging
        self._log_dict({
            'train/mean_reward': mean_reward,
            'train/policy_loss': mean_policy_loss,
            'train/value_loss': mean_value_loss,
            'train/entropy': mean_entropy,
            'train/kl_divergence': mean_kl,
            'train/explained_variance': explained_var
        }, prog_bar=True)
        
        self._log_dict({
            'train/approx_kl': mean_approx_kl,
            'train/clip_fraction': mean_clip_fraction,
            'train/advantage_mean': advantage_mean,
            'train/advantage_std': advantage_std,
            'train/value_mean': value_mean,
            'train/returns_mean': returns_mean,
        }, prog_bar=False)

        if mean_reward is not None and mean_reward >= self.reward_threshold:
            print(f"Early stopping at epoch {self.current_epoch} with mean reward {mean_reward:.2f} >= threshold {self.reward_threshold}")
            self.trainer.should_stop = True

        return mean_policy_loss + mean_value_loss

    def configure_optimizers(self):
        return [
            torch.optim.Adam(self.policy_model.parameters(), lr=self.policy_lr),
            torch.optim.Adam(self.value_model.parameters(), lr=self.value_lr)
        ]

    def _log_dict(self, dict, **kwargs):
        _dict = {k: v for k, v in dict.items() if v is not None}
        self.log_dict(_dict, **kwargs)

    def _evaluate_model(self):
        eval_seed = np.random.randint(0, 1_000_000)
        eval_env = build_env(eval_seed)
        self.policy_model.eval()
        try: 
            return self.__evaluate_model(env)
        finally: 
            self.policy_model.train()
            eval_env.close()

    def __evaluate_model(self, env):
        trajectories = collect_rollouts(
            env,
            self.policy_model,
            self.value_model,
            n_episodes=self.eval_episodes,
            deterministic=True
        )
        episodes = group_trajectories_by_episode(trajectories)
        mean_reward = np.mean([sum(step[2] for step in episode) for episode in episodes])
        return mean_reward
    
    def _collect_and_store_rollout(self):        
        # Collect rollouts and add them to 
        # dataset for sampling during training step
        trajectories = collect_rollouts(
            self.env,
            self.policy_model,
            self.value_model,
            n_steps=2048
        )
        self.rollout_ds.update(*trajectories)

        # Add mean episode rewards to deque 
        # (to be able to average over N episodes)
        episodes = group_trajectories_by_episode(trajectories)
        episode_rewards = [sum(step[2] for step in episode) for episode in episodes]
        for r in episode_rewards: self.episode_reward_deque.append(float(r))
    
    # ---------------------------------------------------
    def forward(self, x):
        return self.policy_model(x)


## 4. Train PPO Agent
We will train the PPO agent on CartPole-v1.

In [8]:
from pytorch_lightning import Trainer
from pytorch_lightning.loggers import WandbLogger

# Create PPO agent and move to device
obs_dim = env.observation_space.shape[0]
act_dim = env.action_space.n if hasattr(env.action_space, 'n') else env.action_space.shape[0]  # Handle discrete and continuous actions
ppo_agent = PPOAgent(obs_dim, act_dim, CONFIG)

# Set up trainer with proper device configuration
wandb_logger = WandbLogger(project="gymnasium_ppo") # TODO: softcode this

trainer = Trainer(
    logger=wandb_logger,
    max_epochs=CONFIG['max_epochs'],
    log_every_n_steps=1,
    enable_progress_bar=True,
    accelerator="auto"
)

# Fit the model
trainer.fit(ppo_agent)

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
wandb: Currently logged in as: tsilva to https://api.wandb.ai. Use `wandb login --relogin` to force relogin



  | Name         | Type      | Params | Mode 
---------------------------------------------------
0 | policy_model | PolicyNet | 4.6 K  | train
1 | value_model  | ValueNet  | 4.5 K  | train
---------------------------------------------------
9.2 K     Trainable params
0         Non-trainable params
9.2 K     Total params
0.037     Total estimated model params size (MB)
14        Modules in train mode
0         Modules in eval mode
/Users/tsilva/repos/tsilva/aiml-notebooks/.venv/lib/python3.12/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:425: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


PPO training started at 2025-07-02 17:02:15
Epoch 9: 100%|██████████| 256/256 [00:25<00:00, 10.06it/s, v_num=bift, train/mean_reward=302.0, train/policy_loss=-0.281, train/value_loss=18.20, train/entropy=0.580, train/kl_divergence=0.00186, train/explained_variance=0.532]
PPO training completed in 199.39 seconds (3.32 minutes)


MisconfigurationException: You can't `self.log()` inside `on_fit_end`. HINT: You can still log directly to the logger by using `self.logger.experiment`.

In [ ]:
import random
from tsilva_notebook_utils.gymnasium import render_episode_frames

n_episodes = 8
trajectories = collect_rollouts(
    build_env(random.randint(0, 1_000_000), n_envs=n_episodes),
    ppo_agent.policy_model,
    n_episodes=n_episodes,
    deterministic=True,
    collect_frames=True
)
episodes = group_trajectories_by_episode(trajectories) # something is wrong in frame collection
mean_reward = np.mean([sum(step[2] for step in episode) for episode in episodes])
episode_frames = [[step[-1] for step in episode] for episode in episodes]
print(f"Mean reward: {mean_reward:.2f}")
render_episode_frames(episode_frames, out_dir="./tmp", grid=(2, 2), text_color=(0, 0, 0))